## Purpose of this notebook is to develop a simple example of Monte Carlo simulation for long range capital planning.

Example Problem:
	
    Suppose a project costs $1 million and has average annual cash flows of $250,000/year with a standard deviation of $100,000. If the cost of capital is %15, evaluate the project.

	If CFs are normally distributed then
		- 68% of the time $150k to $350k
		- 95% of the time $50k to $450k
		- 99% of the time -$50 to $550k

## The Basic Model

We should first build a model that handles non-variable inputs. Once that is working correctly, we can run Monte Carlo simulations using the model. In this model, we want to calculate Net Present Value (NPV) and Internal Rate of Return (IRR) using a mean cash flow (CF). Then later we can intergrate a standard deviation for the CF throughout the 10 year period.

### Capital Budget using NVP & IRR

1. Net Present Value (NPV)

#### What NPV tells you

**Net Present Value (NPV)** measures how much value a project adds (or destroys) in **today’s dollars**.
- **NPV > 0** → project creates value 
- **NPV = 0** → break-even versus required return 
- **NPV < 0** → project destroys value 

#### NPV Formula


$\text{NPV} = \sum_{t=0}^{T} \frac{CF_t}{(1 + r)^t}$


#### Where:

- $ {CF_t}$ = cash flow at time t
- r = discount rate (often WACC or hurdle rate)
- t = 0 is today (usually the initial investment, negative)

2. Internal Rate of Return (IRR)

#### What IRR tells you

**Internal Rate of Return (IRR)** is the **discount rate that makes NPV = 0**.

#### Decision rule:

- **IRR > WACC (or hurdle rate)** → accept 
- **IRR < WACC** → reject 

#### IRR Formula (Conceptual)

Solve for r:


$0 = \sum_{t=0}^{T} \frac{CF_t}{(1 + r)^t}$

In [ ]:
# import dependencies
import numpy as np
import numpy_financial as npf

In [ ]:
def npv(discount_rate, cash_flows):
    """
    Calculate Net Present Value (NPV).

    Parameters
    ----------
    discount_rate : float
        Discount rate (e.g., WACC or hurdle rate), expressed as a decimal.
    cash_flows : list or array
        Cash flows where index 0 is today (t=0).

    Returns
    -------
    float
        Net Present Value
    """
    cash_flows = np.array(cash_flows)
    periods = np.arange(len(cash_flows))
    return np.sum(cash_flows / (1 + discount_rate) ** periods)

In [ ]:
# Calculate NPV using average cash flow
cash_flows = [-1000000,250000,250000,250000,250000,250000,250000,250000,250000,250000,250000]
discount_rate = 0.15

npv_value = npv(discount_rate, cash_flows)

npv_value

In [ ]:
def irr(cash_flows):
    """
    Calculate Internal Rate of Return (IRR).

    Parameters
    ----------
    cash_flows : list or array
        Cash flows where index 0 is today (t=0).

    Returns
    -------
    float
        Internal Rate of Return
    """
    return npf.irr(cash_flows)

In [ ]:
# Calculate IRR
irr(cash_flows)


Using the average cash flow the NPV is greater then 0, so the project creates great value. Given  the Wacc of %15, which is less then our 21% IRR, we can again expect the project to add great value.

### Monte Carlo Simulation

We'll use the fact that the average CF is $250k with a standard deviation of $100k to compute our project risk using Monte Carlo simulation.

### Draws inputs from the distribution 

So here we are concerned with assigning the WACC random value from a distribution to calualte the CF over each year.

In [ ]:
import random

mean_cf = 250000
stdev_cf = 100000

random.normalvariate(mean_cf, stdev_cf)

We can run the code multiple times to see it will choose a random number from our distribution each time, the value represents the respective years cash flow (CF).

### Run the model

Using the same logic as above, will run the model with random inputs.

In [ ]:
random_cash_flows = [-1000000]
yrs = 10

for yr in range(yrs):
    random_cash_flows.append(random.normalvariate(mean_cf, stdev_cf))


# List of random cashflow for each year.
random_cash_flows

In [ ]:
# integrating into NPV calculation
npv_value = npv(discount_rate, random_cash_flows)
npv_value

In [ ]:
# Calculate IRR
irr(random_cash_flows)

Create model and save outputs

In [ ]:
import random

def calc_10_cash_flow(mean_cf, stdev_cf, initial_investment=-100000, yrs=10):
    """
    Generate a list of cash flows:
    - Year 0: initial investment (negative)
    - Years 1–yrs: normally distributed random cash flows
    """
    random_cash_flows = [initial_investment]

    for yr in range(yrs):
        random_cash_flows.append(random.normalvariate(mean_cf, stdev_cf))

    return random_cash_flows

In [ ]:
# iterative model created
iter = 3
mean_cf = 250000
stdev_cf = 100000
outputs = []

for i in range(iter):
    random_cash_flows = calc_10_cash_flow(mean_cf, stdev_cf, initial_investment=-1000000, yrs=10)
    outputs.append([npv(discount_rate, random_cash_flows), irr(random_cash_flows)])

formatted_outputs = [[round(float(a), 2), round(float(b), 2)] for a, b in outputs]
formatted_outputs
    

In [ ]:
### create a function for this
def run_iterations(iterations, mean_cf, stdev_cf, initial_investment, discount_rate, yrs=10):
    outputs = []

    for _ in range(iterations):
        random_cash_flows = calc_10_cash_flow(mean_cf, stdev_cf, initial_investment, yrs)
        outputs.append([npv(discount_rate, random_cash_flows),
                        irr(random_cash_flows)])

    # Format all values to two decimals
    formatted_outputs = [[round(float(a), 2), round(float(b), 2)] 
                         for a, b in outputs]

    return formatted_outputs

In [ ]:
results = run_iterations(
    iterations=1000,
    mean_cf=250000,
    stdev_cf=100000,
    initial_investment=-1000000,
    discount_rate=0.1,
    yrs=10
)


print("1000 iterations ran viewing 5" \
"" \
"")
results[:5]

In [ ]:
import matplotlib.pyplot as plt

def analyze_outputs_matplotlib(formatted_outputs):
    # Extract columns
    first_vals = np.array([row[0] for row in formatted_outputs], dtype=float)
    second_vals = np.array([row[1] for row in formatted_outputs], dtype=float)

    stats = {
        "mean_first": round(first_vals.mean(), 2),
        "mean_second": round(second_vals.mean(), 2),
        "std_first": round(first_vals.std(), 2),
        "std_second": round(second_vals.std(), 2),
        "min_first": round(first_vals.min(), 2),
        "max_first": round(first_vals.max(), 2),
        "min_second": round(second_vals.min(), 2),
        "max_second": round(second_vals.max(), 2),
        "p10_first": round(np.percentile(first_vals, 10), 2),
        "p50_first": round(np.percentile(first_vals, 50), 2),
        "p90_first": round(np.percentile(first_vals, 90), 2),
        "p10_second": round(np.percentile(second_vals, 10), 2),
        "p50_second": round(np.percentile(second_vals, 50), 2),
        "p90_second": round(np.percentile(second_vals, 90), 2),
    }

    # Histogram using matplotlib
    plt.figure(figsize=(10, 6))
    plt.hist(first_vals, bins=40, alpha=0.75, edgecolor='black')
    plt.title("Histogram of First Entries (e.g., NPV)")
    plt.xlabel("First Entry Values")
    plt.ylabel("Frequency")
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    return stats

In [ ]:
stats = analyze_outputs_matplotlib(formatted_outputs)
stats

In [ ]:
plt.show()